# 00 — Puesta en marcha y credenciales

Qué se comprueba acá:

1. Que el paquete importa.
2. Que las credenciales llegan (del `.env` o pasadas a mano).
3. Que el broker responde de verdad — una llamada pública y una firmada.

**Antes de nada**: rellena `BITUNIX_API_KEY` / `BITUNIX_API_SECRET` en el
`.env` de la raíz del repo (ver `.env.example`).

⚠ **Bitunix restringe las API keys por IP.** Si esta máquina no está en la
whitelist de la key, toda llamada *firmada* falla con `10004 Invalid IP` —
los precios y las velas seguirán funcionando porque son públicos. La IP a
poner en la whitelist es la que devuelve
`curl https://fapi.bitunix.com/cdn-cgi/trace` (campo `ip=`), que **no** tiene
por qué coincidir con la de un "what is my IP" cualquiera.

⚠ **Bitunix no tiene testnet.** Lo que se coloque con esas claves es dinero
real. Binance sí la tiene, con claves distintas
(`BINANCE_TEST_API_KEY`, https://testnet.binance.vision).

## Credenciales: de dónde salen

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

from luarix_exchange.brokers.credentials import BrokerCredentials, load_dotenv

leidas = load_dotenv()
print("variables leidas del .env:", sorted(leidas))

for nombre in ("BINANCE", "BITUNIX"):
    c = BrokerCredentials.from_env(nombre)
    # Nunca imprimas la clave entera en un cuaderno: se queda guardada en el
    # fichero y viaja con el a cualquier sitio donde lo abras.
    print(f"{nombre:8} configurado={bool(c)}  testnet={c.has_testnet}  "
          f"key={c.api_key[:6] + '...' if c.api_key else '(vacia)'}")

## Construir el broker

`BrokerFactory` resuelve las credenciales en este orden:

1. El objeto `credentials=` que se le pase — **el caso de producción**: el
   backend tiene una clave por usuario y la pasa explícita.
2. Si no hay, el entorno / `.env` — el caso de las pruebas locales.
3. Si tampoco, levanta con un mensaje que dice qué variable falta.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))   # el repo, sin instalarlo

from luarix_exchange.brokers.factory import AvailableBrokers, BrokerFactory
from luarix_exchange.brokers.credentials import BrokerCredentials, load_dotenv

load_dotenv()   # lee el .env de la raiz del repo

# Cambia esto para probar el otro broker. Todo lo demas del cuaderno
# funciona igual: esa es justamente la idea del contrato unico.
BROKER   = AvailableBrokers.BITUNIX     # o AvailableBrokers.BINANCE
MARKET   = "futures"                    # o "spot"
SYMBOL   = "BTCUSDT"

# Sin `credentials=`, la factory las lee del .env.
# En produccion el backend las pasa explicitas, una por usuario:
#     BrokerFactory.get_broker(BROKER, credentials=BrokerCredentials(
#         api_key=fila.api_key, api_secret=fila.api_secret))
broker = BrokerFactory.get_broker(
    BROKER,
    testnet=False,          # Bitunix lo ignora: no tiene testnet
    test_order=False,       # solo Binance: True valida sin ejecutar
    trading_mode="live",
    allow_live_trading=False,   # sigue bloqueando el place_order real
)
print(type(broker).__name__, "| futuros:", broker.supports_futures, "| OCO:", broker.supports_oco)

## Comprobar que responde

Dos llamadas distintas a propósito:

- `get_ticker` es **pública** — funciona sin credenciales válidas.
- `get_balance` va **firmada** — es la que falla si la key está mal o la IP
  no está en la whitelist.

Si la primera va y la segunda no, el problema son las credenciales o la IP,
no la conexión.

In [ ]:
print("precio publico:", broker.get_ticker(SYMBOL, market=MARKET))

In [ ]:
try:
    saldo = broker.get_balance(market=MARKET, asset="USDT")
    print("llamada firmada OK ->", saldo)
except Exception as e:
    print(type(e).__name__, ":", e)

## Pasar credenciales a mano (lo que hará el backend)

```python
creds = BrokerCredentials(api_key="...", api_secret="...")
broker = BrokerFactory.get_broker(AvailableBrokers.BITUNIX, credentials=creds)
```

Cada usuario del backend tendrá las suyas; la factory no guarda ningún
estado global, así que se pueden tener varios brokers vivos a la vez, uno
por usuario.

In [ ]:
from luarix_exchange.brokers.factory import BrokerFactory, AvailableBrokers
from luarix_exchange.brokers.credentials import BrokerCredentials

# Ejemplo con credenciales explicitas (rellena para probarlo de verdad).
otras = BrokerCredentials(api_key="", api_secret="")
print("¿utilizables?", bool(otras))